## __Tópicos avanzados en Inteligencia Artificial 1 - MIA__

__Profesor__: Anthony D. Cho

__Ayudante__: Luis Oliveros

**Asunto**: Keras-Tensorflow. Uso de Sequential
*****

## Librerias

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from time import time
from numpy import argmin, argmax
from pandas import read_csv
import matplotlib.pyplot as plt
%matplotlib inline

## Pre-processing
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

## Metrics
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

## Keras from tensorflow
from tensorflow.keras.models import Sequential
from tensorflow.keras import layers

## Optimizer
from tensorflow.keras.optimizers import Adam

### Funciones personalizadas

In [ ]:
def plot_history(history, width=12, height=6):
  """
  DESCRIPTION:
    History performance of the keras model
  
  INPUT:
    @param history: history of performance of fitted model
    @type history: tensorflow.python.keras.callbacks.History

  OUTPUT:
    A graphic
  """

  ## Metrics keys stored in tensorflow object
  keys = list(history.history.keys())

  ## Number of epoch used for fit the model
  epoch = range(1, len(history.epoch) +1)

  ## Check if validation set was used.
  withValidation = False
  for key in keys:
    if 'val' in key:
      withValidation = True

  ## Number of metrics 
  nMetrics = len(keys)
  if withValidation:
    nMetrics = nMetrics//2

  ## Plot-space instance
  plt.figure(figsize=(width, height))

  for i in range(nMetrics):
    plt.subplot(nMetrics, 1, i+1)

    ## Plot (train) metric value
    labelMetric = keys[i]
    metric = history.history[keys[i]]
    plt.plot(epoch, metric, 'o-', label=labelMetric)

    if withValidation:
      ## Plot (validation) metric value
      labelMetricVal = keys[i+nMetrics]
      metricVal = history.history[keys[i+nMetrics]]
      plt.plot(epoch, metricVal, 'o-', label=labelMetricVal)

    plt.xlim(epoch[0], epoch[-1])
    plt.legend()
    plt.grid()

  plt.xlabel('Epoch')
  plt.show()

## Dataset

<center>
    <img src=https://www.ganeshdiagnostic.com/admin/public/assets/images/blog/banner/mobile/1701087153-All%20About%20Breast%20Cancer%20in%20women.webp width=800>
</center>

El conjunto de datos contiene información de 10 características de controles clínicos, específicamente de analisis de sangre, de 64 pacientes con cancer de mama y 52 pacientes sanos. 

Los 10 predictores son cuantitativos y la variable de salida (**Classification**) está etiquetada como: 1. Pacienciente sano y 2. Paciente con presencia de cancer

* La data y detalles está completamente disponible en [UCI Repository: Breast Cancer Coimbra](https://archive.ics.uci.edu/ml/datasets/Breast+Cancer+Coimbra)

#### Carga de datos

In [ ]:
## Load data
data = read_csv('https://raw.githubusercontent.com/adoc-box/Datasets/main/breast_cancer_coimbra.csv')

## Feature names list
feature_names = data.columns[:-1]
feature_names

In [ ]:
data.head(4)

In [ ]:
data['Classification'].value_counts()

## Preprocesamiento de datos

In [ ]:
## Predictors and target assignment
X = data.drop(columns=['Classification'])
y = data['Classification']

## Partition sets
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=84)

## scaling
scale = MinMaxScaler().fit(X_train)
X_train = scale.transform(X_train)
X_test = scale.transform(X_test)

## Display data shape
print('(train shape) X: {}, y: {}'.format(X_train.shape, y_train.shape))
print('(test shape) X: {}, y: {}'.format(X_test.shape, y_test.shape))

In [ ]:
## Objetivo target: detectar pacientes con presencia de cancer. 
y_train = y_train-1
y_test = y_test-1

### Diseño del modelo

In [ ]:
## Model instance
model = Sequential(name='Classification')
model.add(layers.Input(shape=(X_train.shape[1],), 
                       name='Input' ))
model.add(layers.Dense(units=64, 
                       activation='relu', 
                       name='Dense_1'))
model.add(layers.Dense(units=16, 
                       activation='relu',
                       name='Dense_2'))
model.add(layers.Dense(units=1,
                       activation='sigmoid',
                       name='output'))
model.summary()

In [ ]:
start = time()

## Compiler setting
model.compile(optimizer=Adam(learning_rate=0.001), 
              loss='binary_crossentropy', 
              metrics=['accuracy'])

## model fitting
history = model.fit(x=X_train, y=y_train, 
                    validation_split=0.15,
                    epochs=200, 
                    batch_size=15)

stop = time()
print('Time spent[s]: {:2f}'.format(stop -start))

In [ ]:
plot_history(history, width=14)

In [ ]:
## Searching for the best epoch
id_min = argmin(history.history['val_loss'])
print('Loss - Validation: {} - Error: {}'.format(id_min+1, history.history['val_loss'][id_min]))

id_max = argmax(history.history['val_accuracy'])
print('Accuracy - Validation: {} - Error: {}'.format(id_max+1, history.history['val_accuracy'][id_max]))

## Mejor modelo

In [ ]:
## Model instance
model = Sequential(name='Classification')
model.add(layers.Input( shape=(X_train.shape[1],), 
                       name='Input' ))
model.add(layers.Dense(units=64, 
                       activation='relu', 
                       name='Dense_1'))
model.add(layers.Dense(units=16, 
                       activation='relu',
                       name='Dense_2'))
model.add(layers.Dense(units=1,
                       activation='sigmoid',
                       name='output'))

## Compiler setting
model.compile(optimizer=Adam(learning_rate=0.001), 
              loss='binary_crossentropy', 
              metrics=['accuracy'])

In [ ]:
start = time()

## model fitting
history = model.fit(x=X_train, y=y_train, 
                    epochs=52, 
                    batch_size=15)

stop = time()
print('Time spent[s]: {:2f}'.format(stop -start))

In [ ]:
## Compute predictions using test data
prediction = model.predict(X_test)
prediction

In [ ]:
## Decodifying prediction to class
predictionClass = prediction.flatten().round()
predictionClass

In [ ]:
## Display confusion matrix
cm = confusion_matrix(y_true=y_test, y_pred=predictionClass)
CM = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['1','2'])
CM.plot();

In [ ]:
## Display classification report
print(classification_report(y_true=y_test, 
                            y_pred=predictionClass, 
                            target_names=['1', '2']
                            ))